# Umbrales climatologicos q90/q95/q99 a partir de MSWEP

Calculo de percentiles de precipitacion diaria (climatologia de referencia) sobre un area de estudio,
usando MSWEP (Multi-Source Weighted-Ensemble Precipitation).

Convenciones seguidas (ajustar segun necesidad):

- Percentiles calculados solo sobre "dias humedos" (precip >= 1 mm/dia), consistente con la definicion
  de los indices ETCCDI R95pTOT / R99pTOT (Zhang et al., 2011, https://doi.org/10.1002/wcc.147).
- Interpolacion lineal para el percentil empirico (metodo por defecto de `numpy.percentile` / `xarray.quantile`,
  equivalente a Hyndman-Fan tipo 7).
- Periodo base climatologico configurable (por defecto 1980-2019, siguiendo la practica estandar de 30+ anios
  usada en WMO/ETCCDI, aunque WMO recomienda formalmente 1991-2020 como nuevo periodo normal).

Los thresholds resultantes se guardan en NetCDF para ser usados en el notebook de evaluacion.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr


## configuracion (editar segun el dataset y area de estudio)

In [4]:
MSWEP_PATH_PATTERN = "/mnt/d/MSWEP_V280/merged/Daily/2025060.nc"

ds = xr.open_mfdataset(MSWEP_PATH_PATTERN, combine="by_coords", chunks={"time": 365})

In [5]:
ds.variables

Frozen({'precipitation': <xarray.Variable (time: 1, lat: 1800, lon: 3600)> Size: 26MB
dask.array<open_dataset-precipitation, shape=(1, 1800, 3600), dtype=float32, chunksize=(1, 200, 200), chunktype=numpy.ndarray>
Attributes:
    units:    mm/d, 'lon': <xarray.IndexVariable 'lon' (lon: 3600)> Size: 14kB
array([-179.95   , -179.84999, -179.75   , ...,  179.75002,  179.85002,
        179.95   ], shape=(3600,), dtype=float32)
Attributes:
    units:      degrees_east
    long_name:  longitude, 'lat': <xarray.IndexVariable 'lat' (lat: 1800)> Size: 7kB
array([ 89.95    ,  89.85    ,  89.75    , ..., -89.75    , -89.850006,
       -89.95001 ], shape=(1800,), dtype=float32)
Attributes:
    units:      degrees_north
    long_name:  latitude, 'time': <xarray.IndexVariable 'time' (time: 1)> Size: 8B
array(['2025-03-01T00:00:00.000000000'], dtype='datetime64[ns]')
Attributes:
    long_name:  time})

In [ ]:
# --- rutas ---
# patron glob a los archivos diarios de MSWEP (NetCDF), p.ej. "/data/MSWEP/daily/*.nc"
MSWEP_PATH_PATTERN = "/mnt/d/MSWEP_V280/merged/Daily/*.nc"
VAR_NAME = "precipitation"          # nombre de la variable de precip en el dataset MSWEP
OUT_THRESHOLDS_NC = "../data/processed/thresholds_q90_q95_q99.nc"

# --- area de estudio (bounding box, grados) ---
LAT_MIN, LAT_MAX = 5.0, 35.0       # TODO: ajustar a la region de interes
LON_MIN, LON_MAX = -140.0, -60.0     # TODO: ajustar a la region de interes

# --- periodo climatologico de referencia ---
CLIM_START = "2000-01-01"
CLIM_END   = "2024-12-31"

# --- percentiles y definicion de dia humedo ---
QUANTILES = [0.90, 0.95, 0.99]
WET_DAY_THRESHOLD_MM = 1.0          # dias con precip < 1 mm se excluyen del calculo (convencion ETCCDI)


## carga y subseteo de datos

In [8]:
ds = xr.open_mfdataset(
    MSWEP_PATH_PATTERN,
    combine="by_coords",
    parallel=True,          # paraleliza la apertura/decodificación de ~9000 archivos diarios
    chunks={},               # deja que dask use el chunking nativo por archivo (1 archivo = 1 chunk de tiempo)
)

# --- verificación de convención de coordenadas, antes de confiar en el slice ---
print(f"lon: [{float(ds.lon.min())}, {float(ds.lon.max())}], "
      f"lat: [{float(ds.lat.min())}, {float(ds.lat.max())}]")

lat_slice = slice(LAT_MAX, LAT_MIN) if ds.lat[0] > ds.lat[-1] else slice(LAT_MIN, LAT_MAX)

da = ds[VAR_NAME].sel(
    time=slice(CLIM_START, CLIM_END),
    lat=lat_slice,
    lon=slice(LON_MIN, LON_MAX),
)

# filtro estacional ANTES de materializar — reduce el volumen efectivo a leer en ~41.5% (214/365)
da = da.sel(time=da.time.dt.month.isin(range(5, 12)))

# rechunk explícito: un solo chunk a lo largo de 'time' (requerido por dask para .quantile),
# chunks moderados en espacio para no exceder memoria por bloque
da = da.chunk({"time": -1, "lat": 100, "lon": 200})

print(da)
print(f"n_time = {da.sizes['time']}, n_lat = {da.sizes['lat']}, n_lon = {da.sizes['lon']}")
print(f"tamaño estimado en memoria: {da.nbytes / 1e9:.2f} GB")

: 

## calculo de percentiles sobre dias humedos

Se enmascaran los dias secos (precip < `WET_DAY_THRESHOLD_MM`) antes de calcular el percentil, por grid point.

In [ ]:
da_wet = da.where(da >= WET_DAY_THRESHOLD_MM)

# xr.quantile ignora NaN por defecto (skipna=True), asi que los dias secos enmascarados no contribuyen
thresholds = da_wet.chunk({"time": -1}).quantile(QUANTILES, dim="time", skipna=True)
thresholds = thresholds.rename({"quantile": "q"})
thresholds = thresholds.compute()

thresholds


## verificacion rapida

In [ ]:
# chequeo 1: monotonicidad q90 <= q95 <= q99 en cada punto de grilla
q90, q95, q99 = thresholds.sel(q=0.90), thresholds.sel(q=0.95), thresholds.sel(q=0.99)
assert bool((q90 <= q95).where(~np.isnan(q90)).fillna(True).all()), "q90 > q95 en algun punto: revisar"
assert bool((q95 <= q99).where(~np.isnan(q95)).fillna(True).all()), "q95 > q99 en algun punto: revisar"

# chequeo 2: comparar contra numpy.percentile en un punto de grilla aleatorio, como cross-check independiente
i_lat, i_lon = da.sizes["lat"] // 2, da.sizes["lon"] // 2
serie = da.isel(lat=i_lat, lon=i_lon).values
serie_wet = serie[serie >= WET_DAY_THRESHOLD_MM]
np_check = np.nanpercentile(serie_wet, [90, 95, 99])
xr_check = thresholds.isel(lat=i_lat, lon=i_lon).values

print("numpy.percentile :", np_check)
print("xarray.quantile  :", xr_check)
print("diferencia maxima:", np.nanmax(np.abs(np_check - xr_check)))


## mapas de los umbrales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, q in zip(axes, QUANTILES, strict=False):
    thresholds.sel(q=q).plot(ax=ax, cmap="viridis", cbar_kwargs={"label": "mm/dia"})
    ax.set_title(f"q{int(q*100)}")
plt.show()


## guardado

In [ ]:
thresholds.attrs.update({
    "description": "percentiles de precipitacion diaria calculados sobre dias humedos (>=1 mm/dia)",
    "source": "MSWEP",
    "clim_period": f"{CLIM_START} to {CLIM_END}",
    "wet_day_threshold_mm": WET_DAY_THRESHOLD_MM,
})
thresholds.to_netcdf(OUT_THRESHOLDS_NC)
print(f"guardado en {Path(OUT_THRESHOLDS_NC).resolve()}")
